# Case Study 8: Time Series Classification — ECG5000 Heartbeat Classification

## RNN vs LSTM vs GRU Comprehensive Comparison

---

### Objective
Classify heartbeat ECG signals into **5 diagnostic categories** using three recurrent architectures:
- **Vanilla RNN** — simple recurrence, prone to vanishing gradients on long sequences
- **LSTM** — gated memory cells (forget, input, output gates) for long-range dependencies
- **GRU** — simplified gating (reset, update gates) with fewer parameters

### What You Will Learn
1. How to frame time series classification as a sequence-to-label problem
2. Handling severe class imbalance in medical signal data
3. Building and comparing RNN, LSTM, GRU classifiers in PyTorch
4. Evaluating multi-class classification with per-class metrics, confusion matrices, and ROC curves
5. Gradient-based saliency analysis for interpretability on ECG signals

### Dataset — ECG5000
- **Source**: UCR Time Series Classification Archive (Yanping Chen et al., 2015)
- **Domain**: Cardiology — automated heartbeat classification from Electrocardiogram (ECG) recordings
- **Size**: 5,000 heartbeat samples (500 TRAIN + 4,500 TEST)
- **Signal Length**: 140 time steps per heartbeat
- **Classes**: 5 categories
  - **Class 1**: Normal heartbeat (dominant, ~59%)
  - **Class 2**: R-on-T Premature Ventricular Contraction (PVC)
  - **Class 3**: Supraventricular Premature or Ectopic Beat
  - **Class 4**: Premature Ventricular Contraction
  - **Class 5**: Unclassifiable heartbeat

### Medical Relevance
Automated ECG classification is critical in clinical settings:
- **Early detection** of arrhythmias and cardiac abnormalities
- **Continuous monitoring** via wearable devices (smartwatches, Holter monitors)
- **Reducing clinician workload** by triaging normal vs abnormal beats
- **Real-time alerting** in ICU and telemetry units

The challenge here is to accurately classify not just normal beats, but the rare abnormal classes — where clinical value is highest.

---
## 1. Environment Setup

In [ ]:
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc, roc_auc_score
)
from sklearn.manifold import TSNE

# ==================== Reproducibility ====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ==================== Plot Style ====================
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}

# Class names for the 5 ECG categories
CLASS_NAMES = {
    1: 'Normal',
    2: 'R-on-T PVC',
    3: 'SP/Ectopic',
    4: 'PVC',
    5: 'Unclassifiable'
}

CLASS_COLORS = {
    1: '#2ecc71',   # green - Normal
    2: '#e74c3c',   # red - R-on-T PVC
    3: '#3498db',   # blue - SP/Ectopic
    4: '#f39c12',   # orange - PVC
    5: '#9b59b6'    # purple - Unclassifiable
}

print('\nEnvironment ready.')

---
## 2. Data Loading & Exploratory Data Analysis

In [ ]:
# Load space-separated txt files using np.loadtxt()
train_raw = np.loadtxt('Anomaly_Detection_and_Signals/ecg5000/ECG5000_TRAIN.txt')
test_raw = np.loadtxt('Anomaly_Detection_and_Signals/ecg5000/ECG5000_TEST.txt')

print(f'Train shape: {train_raw.shape}')   # (500, 141)
print(f'Test shape:  {test_raw.shape}')     # (4500, 141)

# Combine TRAIN + TEST for a total of 5000 samples
# (We will do our own stratified split later)
data_all = np.concatenate([train_raw, test_raw], axis=0)
print(f'Combined shape: {data_all.shape}')  # (5000, 141)

In [ ]:
# Separate labels (first column) and signals (remaining 140 columns)
labels = data_all[:, 0].astype(int)       # Class labels: 1-5
signals = data_all[:, 1:]                  # ECG signal values: 140 timesteps

print(f'Labels shape:  {labels.shape}')     # (5000,)
print(f'Signals shape: {signals.shape}')    # (5000, 140)
print(f'\nUnique labels: {np.unique(labels)}')
print(f'Signal length: {signals.shape[1]} timesteps')
print(f'\nSignal value range: [{signals.min():.4f}, {signals.max():.4f}]')
print(f'Signal mean: {signals.mean():.4f}, std: {signals.std():.4f}')

In [ ]:
# Class distribution analysis
unique_classes, class_counts = np.unique(labels, return_counts=True)
class_dist = pd.DataFrame({
    'Class': [f'Class {c}: {CLASS_NAMES[c]}' for c in unique_classes],
    'Count': class_counts,
    'Percentage': (class_counts / len(labels) * 100).round(2)
})
print('Class Distribution:')
print('=' * 60)
print(class_dist.to_string(index=False))
print(f'\nImbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x')

In [ ]:
# Class distribution bar chart (show imbalance - class 1 dominates)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
bar_colors = [CLASS_COLORS[c] for c in unique_classes]
bar_labels = [f'Class {c}\n{CLASS_NAMES[c]}' for c in unique_classes]
bars = ax1.bar(bar_labels, class_counts, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, count, pct in zip(bars, class_counts, class_dist['Percentage']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{count}\n({pct}%)', ha='center', fontweight='bold', fontsize=10)
ax1.set_title('ECG5000 Class Distribution', fontsize=14)
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.set_ylim(0, max(class_counts) * 1.15)

# Pie chart
wedges, texts, autotexts = ax2.pie(
    class_counts, labels=[CLASS_NAMES[c] for c in unique_classes],
    colors=bar_colors, autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 10}
)
for autotext in autotexts:
    autotext.set_fontweight('bold')
ax2.set_title('Class Proportions', fontsize=14)

plt.suptitle('Severe Class Imbalance: Normal Beats Dominate (~59%)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Key observation: Class 1 (Normal) dominates with ~59% of samples.')
print('Classes 3, 4, 5 are minority classes with <10% each.')
print('This imbalance must be addressed during training (class weights / balanced sampling).')

In [ ]:
# Average heartbeat signal per class (overlay 5 colored lines on same axes)
fig, ax = plt.subplots(figsize=(14, 6))

timesteps = np.arange(signals.shape[1])

for cls in unique_classes:
    mask = labels == cls
    mean_signal = signals[mask].mean(axis=0)
    std_signal = signals[mask].std(axis=0)
    
    ax.plot(timesteps, mean_signal, color=CLASS_COLORS[cls],
            linewidth=2.0, label=f'Class {cls}: {CLASS_NAMES[cls]} (n={mask.sum()})')
    ax.fill_between(timesteps, mean_signal - std_signal, mean_signal + std_signal,
                    color=CLASS_COLORS[cls], alpha=0.1)

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Signal Amplitude', fontsize=12)
ax.set_title('Average ECG Heartbeat Signal per Class (with +/- 1 std)', fontsize=14)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

print('Observations:')
print('- Normal beats (Class 1) show a clean, well-defined QRS complex')
print('- Abnormal classes show deviations in amplitude, timing, and morphology')
print('- Classes 2 and 4 (PVC variants) have distinctly different waveforms')
print('- The std bands show that some classes have much higher variability')

In [ ]:
# Grid of individual sample heartbeats per class (5x3 subplot grid)
fig, axes = plt.subplots(5, 3, figsize=(16, 14))

for row, cls in enumerate(unique_classes):
    cls_indices = np.where(labels == cls)[0]
    # Pick 3 random samples from each class
    np.random.seed(SEED)
    sample_indices = np.random.choice(cls_indices, size=min(3, len(cls_indices)), replace=False)
    
    for col, idx in enumerate(sample_indices):
        ax = axes[row, col]
        ax.plot(timesteps, signals[idx], color=CLASS_COLORS[cls], linewidth=1.2)
        ax.fill_between(timesteps, 0, signals[idx], alpha=0.15, color=CLASS_COLORS[cls])
        ax.set_ylim(signals.min() * 1.1, signals.max() * 1.1)
        
        if col == 0:
            ax.set_ylabel(f'Class {cls}\n{CLASS_NAMES[cls]}', fontsize=10, fontweight='bold')
        if row == 0:
            ax.set_title(f'Sample {col+1}', fontsize=11)
        if row == 4:
            ax.set_xlabel('Time Step', fontsize=10)
        
        ax.text(0.98, 0.95, f'idx={idx}', transform=ax.transAxes,
                ha='right', va='top', fontsize=8, color='gray')

plt.suptitle('Individual ECG Heartbeat Samples by Class', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Signal statistics per class (mean, std, min, max)
stats_list = []
for cls in unique_classes:
    mask = labels == cls
    cls_signals = signals[mask]
    stats_list.append({
        'Class': f'{cls}: {CLASS_NAMES[cls]}',
        'Count': int(mask.sum()),
        'Mean': cls_signals.mean(),
        'Std': cls_signals.std(),
        'Min': cls_signals.min(),
        'Max': cls_signals.max(),
        'Median': np.median(cls_signals),
        'Skewness': float(pd.Series(cls_signals.flatten()).skew()),
        'Kurtosis': float(pd.Series(cls_signals.flatten()).kurtosis())
    })

stats_df = pd.DataFrame(stats_list)
for col in ['Mean', 'Std', 'Min', 'Max', 'Median', 'Skewness', 'Kurtosis']:
    stats_df[col] = stats_df[col].round(4)

print('Signal Statistics per Class:')
print('=' * 100)
print(stats_df.to_string(index=False))

In [ ]:
# t-SNE visualization of raw signals colored by class
print('Computing t-SNE embedding of raw ECG signals...')
print('(This may take a minute for 5000 samples with 140 dimensions)')

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000, verbose=0)
signals_tsne = tsne.fit_transform(signals)

fig, ax = plt.subplots(figsize=(10, 8))

for cls in unique_classes:
    mask = labels == cls
    ax.scatter(signals_tsne[mask, 0], signals_tsne[mask, 1],
               c=CLASS_COLORS[cls], label=f'Class {cls}: {CLASS_NAMES[cls]}',
               s=15, alpha=0.6, edgecolors='none')

ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('t-SNE of Raw ECG Signals (5000 samples, 140 dims)', fontsize=14)
ax.legend(fontsize=10, markerscale=3, framealpha=0.9)
plt.tight_layout()
plt.show()

print('Observations:')
print('- Normal beats (Class 1) form a large, dense cluster')
print('- Abnormal classes tend to occupy distinct regions in the embedding space')
print('- Some overlap exists between classes, especially minority ones')
print('- This suggests the classification task is feasible but non-trivial')

---
## 3. Preprocessing

In [ ]:
# Stratified train/val/test split (70/15/15)
# Convert labels to 0-indexed for PyTorch (Class 1->0, Class 2->1, ..., Class 5->4)
labels_0idx = labels - 1  # Now 0-4 instead of 1-5

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    signals, labels_0idx, test_size=0.30, random_state=SEED, stratify=labels_0idx
)

# Second split: 50/50 of temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f'Train: {X_train.shape[0]} samples ({X_train.shape[0]/len(labels)*100:.1f}%)')
print(f'Val:   {X_val.shape[0]} samples ({X_val.shape[0]/len(labels)*100:.1f}%)')
print(f'Test:  {X_test.shape[0]} samples ({X_test.shape[0]/len(labels)*100:.1f}%)')

# Verify stratification
print('\nClass distribution per split:')
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(y, return_counts=True)
    dist = ', '.join([f'C{c+1}:{n}' for c, n in zip(unique, counts)])
    print(f'  {name}: {dist}')

In [ ]:
# StandardScaler on signal values (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f'Scaler fitted on {X_train.shape[0]} training samples')
print(f'Train scaled range: [{X_train_scaled.min():.4f}, {X_train_scaled.max():.4f}]')
print(f'Val scaled range:   [{X_val_scaled.min():.4f}, {X_val_scaled.max():.4f}]')
print(f'Test scaled range:  [{X_test_scaled.min():.4f}, {X_test_scaled.max():.4f}]')

In [ ]:
# Reshape for RNN: (batch, 140, 1) - single-feature time series
# Each heartbeat is a sequence of 140 timesteps with 1 feature (amplitude)
X_train_tensor = torch.FloatTensor(X_train_scaled).unsqueeze(-1)  # (N, 140, 1)
X_val_tensor = torch.FloatTensor(X_val_scaled).unsqueeze(-1)
X_test_tensor = torch.FloatTensor(X_test_scaled).unsqueeze(-1)

y_train_tensor = torch.LongTensor(y_train)
y_val_tensor = torch.LongTensor(y_val)
y_test_tensor = torch.LongTensor(y_test)

print(f'X_train_tensor: {X_train_tensor.shape}')  # (3500, 140, 1)
print(f'y_train_tensor: {y_train_tensor.shape}')  # (3500,)
print(f'X_val_tensor:   {X_val_tensor.shape}')    # (750, 140, 1)
print(f'X_test_tensor:  {X_test_tensor.shape}')   # (750, 140, 1)
print(f'\nInput to RNN/LSTM/GRU: (batch, seq_len=140, input_size=1)')

In [ ]:
# Compute class weights (inverse frequency) for weighted loss
train_class_counts = np.bincount(y_train, minlength=5)
class_weights = 1.0 / train_class_counts.astype(np.float32)
class_weights = class_weights / class_weights.sum() * len(train_class_counts)  # Normalize
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

print('Class weights for CrossEntropyLoss (inverse frequency, normalized):')
for i, (cnt, w) in enumerate(zip(train_class_counts, class_weights)):
    print(f'  Class {i+1} ({CLASS_NAMES[i+1]:15s}): count={cnt:4d}, weight={w:.4f}')

In [ ]:
# Custom ECGDataset class and DataLoaders with balanced sampling
class ECGDataset(Dataset):
    """PyTorch Dataset for ECG heartbeat signals."""
    
    def __init__(self, X, y):
        """
        Args:
            X: tensor of shape (N, seq_len, 1) - ECG signals
            y: tensor of shape (N,) - class labels (0-indexed)
        """
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create datasets
train_dataset = ECGDataset(X_train_tensor, y_train_tensor)
val_dataset = ECGDataset(X_val_tensor, y_val_tensor)
test_dataset = ECGDataset(X_test_tensor, y_test_tensor)

# Balanced sampler for training (oversample minority classes)
sample_weights = class_weights[y_train]  # Weight per sample
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'DataLoaders created with batch_size={BATCH_SIZE}')
print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')
print(f'\nUsing WeightedRandomSampler to handle class imbalance during training')

# Quick check: sample a batch
sample_X, sample_y = next(iter(train_loader))
print(f'\nSample batch: X={sample_X.shape}, y={sample_y.shape}')
print(f'Batch class distribution: {np.bincount(sample_y.numpy(), minlength=5)}')

---
## 4. Model Architecture

### Architecture Diagram
```
Input: (batch, seq_len=140, features=1)
       |
  +--------+   +--------+   +--------+         +--------+
  | RNN /  |-->| RNN /  |-->| RNN /  |--> ... ->| RNN /  |--> h_T
  | LSTM / |   | LSTM / |   | LSTM / |         | LSTM / |     |
  | GRU    |   | GRU    |   | GRU    |         | GRU    |  [if bidirectional:
  +--------+   +--------+   +--------+         +--------+   concat h_fwd + h_bwd]
    t=1          t=2          t=3                 t=140        |
                                                         [Linear Layer]
                                                               |
                                                         Output: (batch, 5)
                                                    (logits for 5 ECG classes)
```

### Key Design Decisions
- **Last hidden state** is used as the sequence representation (not attention or pooling)
- **Dropout** between RNN layers (only if `num_layers > 1`)
- **Bidirectional** option: processes sequence both forward and backward
- **Output**: 5-class logits (no softmax — handled by `CrossEntropyLoss`)

In [ ]:
class SignalClassifier(nn.Module):
    """Unified RNN/LSTM/GRU model for ECG signal classification.
    
    Architecture:
        Input (batch, 140, 1) -> RNN/LSTM/GRU -> Last Hidden -> Linear -> 5 classes
    """
    
    SUPPORTED_TYPES = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
    
    def __init__(self, model_type, input_size=1, hidden_size=64, num_classes=5,
                 num_layers=1, dropout=0.0, bidirectional=False):
        super().__init__()
        assert model_type in self.SUPPORTED_TYPES, \
            f'model_type must be one of {list(self.SUPPORTED_TYPES.keys())}'
        
        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        # Recurrent layer
        rnn_cls = self.SUPPORTED_TYPES[model_type]
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        # Classification head
        fc_input_size = hidden_size * self.num_directions
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(fc_input_size, num_classes)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len=140, input_size=1)
        Returns:
            logits: (batch, num_classes=5)
        """
        # RNN output: (batch, seq_len, hidden_size * num_directions)
        rnn_out, _ = self.rnn(x)
        
        # Take the last timestep output
        last_hidden = rnn_out[:, -1, :]  # (batch, hidden_size * num_directions)
        
        # Classification
        out = self.dropout(last_hidden)
        logits = self.fc(out)  # (batch, num_classes)
        return logits
    
    def count_parameters(self):
        """Count trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Quick test
for mt in ['RNN', 'LSTM', 'GRU']:
    model = SignalClassifier(mt, input_size=1, hidden_size=64, num_classes=5, num_layers=2)
    test_input = torch.randn(4, 140, 1)
    test_output = model(test_input)
    print(f'{mt:5s}: input={test_input.shape} -> output={test_output.shape}, params={model.count_parameters():,}')

In [ ]:
# Bidirectional variant demonstration
print('Bidirectional Models:')
print('=' * 60)
for mt in ['RNN', 'LSTM', 'GRU']:
    model_uni = SignalClassifier(mt, hidden_size=64, num_layers=2, bidirectional=False)
    model_bi = SignalClassifier(mt, hidden_size=64, num_layers=2, bidirectional=True)
    print(f'{mt}: Unidirectional={model_uni.count_parameters():,} params, '
          f'Bidirectional={model_bi.count_parameters():,} params '
          f'(+{model_bi.count_parameters() - model_uni.count_parameters():,})')

In [ ]:
# Parameter count comparison bar chart
configs_to_compare = [
    ('RNN\n(uni)', 'RNN', False),
    ('LSTM\n(uni)', 'LSTM', False),
    ('GRU\n(uni)', 'GRU', False),
    ('RNN\n(bi)', 'RNN', True),
    ('LSTM\n(bi)', 'LSTM', True),
    ('GRU\n(bi)', 'GRU', True),
]

fig, ax = plt.subplots(figsize=(12, 5))

names, param_counts, bar_colors = [], [], []
for label, mt, bi in configs_to_compare:
    model = SignalClassifier(mt, hidden_size=64, num_layers=2, bidirectional=bi)
    names.append(label)
    param_counts.append(model.count_parameters())
    bar_colors.append(COLORS[mt])

bars = ax.bar(names, param_counts, color=bar_colors, edgecolor='white', width=0.6,
              alpha=[0.7, 0.7, 0.7, 1.0, 1.0, 1.0])
for bar, val in zip(bars, param_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', fontweight='bold', fontsize=9)

ax.set_title('Parameter Count: Unidirectional vs Bidirectional (hidden=64, layers=2)', fontsize=13)
ax.set_ylabel('Trainable Parameters', fontsize=12)
plt.tight_layout()
plt.show()

print('LSTM has ~4x parameters of RNN (3 extra gates)')
print('GRU has ~3x parameters of RNN (2 gates)')
print('Bidirectional roughly doubles the recurrent parameters')

---
## 5. Training Infrastructure

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, class_weights,
                device=DEVICE, patience=10, clip_grad=1.0, verbose=True):
    """Train a classification model with weighted loss, early stopping, gradient clipping.
    
    Args:
        model: SignalClassifier instance
        train_loader: DataLoader for training
        val_loader: DataLoader for validation
        epochs: max epochs
        lr: learning rate
        class_weights: tensor of class weights for CrossEntropyLoss
        device: torch device
        patience: early stopping patience
        clip_grad: max gradient norm
        verbose: print progress
    
    Returns:
        history: dict with train_loss, val_loss, val_accuracy, val_macro_f1
        training_time: seconds
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5, verbose=False
    )
    
    history = {
        'train_loss': [], 'val_loss': [],
        'val_accuracy': [], 'val_macro_f1': []
    }
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    start_time = time.time()
    
    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_losses.append(loss.item())
        
        # ---- Validation ----
        model.eval()
        val_losses = []
        all_preds, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())
                preds = logits.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())
        
        avg_train = np.mean(train_losses)
        avg_val = np.mean(val_losses)
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['val_accuracy'].append(val_acc)
        history['val_macro_f1'].append(val_f1)
        
        scheduler.step(avg_val)
        
        # Early stopping
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1}')
                break
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | '
                  f'Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | '
                  f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')
    
    training_time = time.time() - start_time
    
    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(device)
    
    return history, training_time

In [ ]:
def evaluate_classifier(model, data_loader, device=DEVICE, return_probs=False):
    """Evaluate a classifier and return comprehensive metrics.
    
    Args:
        model: trained SignalClassifier
        data_loader: DataLoader to evaluate on
        device: torch device
        return_probs: if True, also return predicted probabilities
    
    Returns:
        metrics: dict with accuracy, macro_f1, weighted_f1, per-class metrics
        cm: confusion matrix
        preds: numpy array of predictions
        probs: numpy array of probabilities (if return_probs=True)
    """
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Overall metrics
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
    cm = confusion_matrix(all_labels, all_preds)
    
    # Per-class F1
    per_class_f1 = f1_score(all_labels, all_preds, average=None)
    
    metrics = {
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'per_class_f1': per_class_f1
    }
    
    if return_probs:
        return metrics, cm, all_preds, all_probs, all_labels
    return metrics, cm, all_preds

In [ ]:
def plot_confusion_matrices(cms, model_names, class_names_list, figsize=(18, 5)):
    """Plot side-by-side confusion matrices."""
    fig, axes = plt.subplots(1, len(cms), figsize=figsize)
    if len(cms) == 1:
        axes = [axes]
    
    for ax, cm, name in zip(axes, cms, model_names):
        # Normalize
        cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=class_names_list, yticklabels=class_names_list,
                    cbar_kws={'shrink': 0.8})
        ax.set_title(f'{name}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=11)
        ax.set_ylabel('Actual', fontsize=11)
    
    plt.suptitle('Confusion Matrices (Test Set)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


print('Training infrastructure ready.')
print('Functions defined: train_model(), evaluate_classifier(), plot_confusion_matrices()')

---
## 6. Hyperparameter Tuning

We perform a **random search** over key hyperparameters for each model type (RNN, LSTM, GRU).

- 15 random configurations sampled from the search space
- Each configuration is evaluated for all 3 model types = **45 total trials**
- Quick training (30 epochs) to find the best hyperparameters efficiently

In [ ]:
# Search space
SEARCH_SPACE = {
    'hidden_size': [32, 64, 128, 256],
    'num_layers': [1, 2, 3],
    'lr': [1e-4, 5e-4, 1e-3, 5e-3],
    'dropout': [0.0, 0.1, 0.2, 0.3],
    'bidirectional': [True, False]
}

NUM_RANDOM_CONFIGS = 15
TUNING_EPOCHS = 30

# Generate 15 random configs
random.seed(SEED)
random_configs = []
for i in range(NUM_RANDOM_CONFIGS):
    config = {
        'hidden_size': random.choice(SEARCH_SPACE['hidden_size']),
        'num_layers': random.choice(SEARCH_SPACE['num_layers']),
        'lr': random.choice(SEARCH_SPACE['lr']),
        'dropout': random.choice(SEARCH_SPACE['dropout']),
        'bidirectional': random.choice(SEARCH_SPACE['bidirectional'])
    }
    random_configs.append(config)

print(f'Search space:')
for k, v in SEARCH_SPACE.items():
    print(f'  {k}: {v}')
print(f'\nGenerated {NUM_RANDOM_CONFIGS} random configs x 3 model types = {NUM_RANDOM_CONFIGS * 3} total trials')
print(f'Tuning epochs per trial: {TUNING_EPOCHS}')

print('\nSample configs:')
for i, cfg in enumerate(random_configs[:5]):
    print(f'  Config {i+1}: {cfg}')

In [ ]:
# Run hyperparameter search
tuning_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*70}')
    print(f'Tuning {model_type} ({NUM_RANDOM_CONFIGS} configs, {TUNING_EPOCHS} epochs each)')
    print(f'{"="*70}')
    
    for i, config in enumerate(random_configs):
        # Build model
        model = SignalClassifier(
            model_type=model_type,
            input_size=1,
            hidden_size=config['hidden_size'],
            num_classes=5,
            num_layers=config['num_layers'],
            dropout=config['dropout'],
            bidirectional=config['bidirectional']
        )
        
        # Train
        history, train_time = train_model(
            model, train_loader, val_loader,
            epochs=TUNING_EPOCHS, lr=config['lr'],
            class_weights=class_weights_tensor,
            patience=8, verbose=False
        )
        
        best_val_loss = min(history['val_loss'])
        best_val_acc = max(history['val_accuracy'])
        best_val_f1 = max(history['val_macro_f1'])
        
        tuning_results.append({
            'model_type': model_type,
            'config_id': i,
            **config,
            'best_val_loss': best_val_loss,
            'best_val_acc': best_val_acc,
            'best_val_f1': best_val_f1,
            'train_time': train_time,
            'epochs_run': len(history['val_loss']),
            'params': model.count_parameters()
        })
        
        bi_str = 'Bi' if config['bidirectional'] else 'Uni'
        print(f'  [{i+1:2d}/{NUM_RANDOM_CONFIGS}] h={config["hidden_size"]:3d}, '
              f'L={config["num_layers"]}, lr={config["lr"]:.4f}, '
              f'drop={config["dropout"]:.1f}, {bi_str:3s} '
              f'-> F1={best_val_f1:.4f}, Acc={best_val_acc:.4f} ({train_time:.1f}s)')

tuning_df = pd.DataFrame(tuning_results)
print(f'\nTotal trials completed: {len(tuning_df)}')

In [ ]:
# Find best config per model type (by val macro F1)
best_configs = {}
print('Best Configurations per Model Type (by Val Macro-F1):')
print('=' * 90)

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    best_row = subset.loc[subset['best_val_f1'].idxmax()]
    best_configs[model_type] = best_row.to_dict()
    bi_str = 'Bidirectional' if best_row['bidirectional'] else 'Unidirectional'
    print(f'\n{model_type}:')
    print(f'  hidden_size={int(best_row["hidden_size"])}, num_layers={int(best_row["num_layers"])}, '
          f'lr={best_row["lr"]}, dropout={best_row["dropout"]}, {bi_str}')
    print(f'  Val Macro-F1: {best_row["best_val_f1"]:.4f}, '
          f'Val Accuracy: {best_row["best_val_acc"]:.4f}, '
          f'Params: {int(best_row["params"]):,}')

In [ ]:
# Visualize tuning results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    subset = tuning_df[tuning_df['model_type'] == model_type].copy()
    subset['bi_label'] = subset['bidirectional'].map({True: 'Bi', False: 'Uni'})
    
    scatter = ax.scatter(
        subset['params'], subset['best_val_f1'],
        c=subset['hidden_size'], cmap='viridis',
        s=80, alpha=0.7, edgecolors='white', linewidth=0.5
    )
    
    # Mark best
    best_idx = subset['best_val_f1'].idxmax()
    ax.scatter(subset.loc[best_idx, 'params'], subset.loc[best_idx, 'best_val_f1'],
               s=200, facecolors='none', edgecolors='red', linewidths=2.5,
               label='Best', zorder=5)
    
    ax.set_xlabel('Parameters', fontsize=11)
    ax.set_ylabel('Val Macro-F1', fontsize=11)
    ax.set_title(f'{model_type}', fontsize=13, color=COLORS[model_type], fontweight='bold')
    ax.legend(fontsize=9)
    plt.colorbar(scatter, ax=ax, label='hidden_size', shrink=0.8)

plt.suptitle('Hyperparameter Tuning: F1 vs Model Size', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top-5 configs per model type
print('Top-5 Configurations per Model Type:')
print('=' * 100)

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    top5 = subset.nlargest(5, 'best_val_f1')[[
        'hidden_size', 'num_layers', 'lr', 'dropout', 'bidirectional',
        'best_val_f1', 'best_val_acc', 'params'
    ]]
    print(f'\n{model_type}:')
    print(top5.to_string(index=False))

---
## 7. Final Training & Comparison

In [ ]:
# Retrain best models with more epochs (80 epochs)
FINAL_EPOCHS = 80
final_models = {}
final_histories = {}
final_times = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*60}')
    print(f'Training final {model_type} model ({FINAL_EPOCHS} epochs)')
    print(f'{"="*60}')
    cfg = best_configs[model_type]
    
    model = SignalClassifier(
        model_type=model_type,
        input_size=1,
        hidden_size=int(cfg['hidden_size']),
        num_classes=5,
        num_layers=int(cfg['num_layers']),
        dropout=cfg['dropout'],
        bidirectional=cfg['bidirectional']
    )
    
    print(f'  Config: hidden={int(cfg["hidden_size"])}, layers={int(cfg["num_layers"])}, '
          f'lr={cfg["lr"]}, dropout={cfg["dropout"]}, '
          f'bi={cfg["bidirectional"]}, params={model.count_parameters():,}')
    
    history, train_time = train_model(
        model, train_loader, val_loader,
        epochs=FINAL_EPOCHS, lr=cfg['lr'],
        class_weights=class_weights_tensor,
        patience=15, verbose=True
    )
    
    final_models[model_type] = model
    final_histories[model_type] = history
    final_times[model_type] = train_time
    
    print(f'  Done: {len(history["val_loss"])} epochs, {train_time:.1f}s')

In [ ]:
# Training curves overlay (loss + accuracy)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for model_type in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[model_type]
    epochs_range = range(1, len(h['train_loss']) + 1)
    
    axes[0, 0].plot(epochs_range, h['train_loss'], color=COLORS[model_type],
                     label=model_type, linewidth=1.5)
    axes[0, 1].plot(epochs_range, h['val_loss'], color=COLORS[model_type],
                     label=model_type, linewidth=1.5)
    axes[1, 0].plot(epochs_range, h['val_accuracy'], color=COLORS[model_type],
                     label=model_type, linewidth=1.5)
    axes[1, 1].plot(epochs_range, h['val_macro_f1'], color=COLORS[model_type],
                     label=model_type, linewidth=1.5)

titles = ['Training Loss', 'Validation Loss', 'Validation Accuracy', 'Validation Macro-F1']
ylabels = ['Loss', 'Loss', 'Accuracy', 'Macro-F1']

for ax, title, ylabel in zip(axes.flat, titles, ylabels):
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=10)

plt.suptitle('Training Curves: RNN vs LSTM vs GRU', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate all models on test set
all_metrics = {}
all_cms = {}
all_preds_dict = {}
all_probs_dict = {}
all_test_labels = None

print('Test Set Evaluation:')
print('=' * 70)

for model_type in ['RNN', 'LSTM', 'GRU']:
    metrics, cm, preds, probs, test_labels = evaluate_classifier(
        final_models[model_type], test_loader, return_probs=True
    )
    all_metrics[model_type] = metrics
    all_cms[model_type] = cm
    all_preds_dict[model_type] = preds
    all_probs_dict[model_type] = probs
    all_test_labels = test_labels
    
    print(f'\n{model_type}:')
    print(f'  Accuracy:    {metrics["accuracy"]:.4f}')
    print(f'  Macro-F1:    {metrics["macro_f1"]:.4f}')
    print(f'  Weighted-F1: {metrics["weighted_f1"]:.4f}')
    print(f'  Per-class F1: {["{:.3f}".format(f) for f in metrics["per_class_f1"]]}')

In [ ]:
# Confusion matrices side-by-side (3 heatmaps with class labels)
short_names = [f'C{i+1}' for i in range(5)]
plot_confusion_matrices(
    [all_cms['RNN'], all_cms['LSTM'], all_cms['GRU']],
    ['RNN', 'LSTM', 'GRU'],
    short_names,
    figsize=(20, 5)
)

In [ ]:
# Per-class F1 grouped bar chart (5 classes x 3 models)
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(5)  # 5 classes
width = 0.25

for i, model_type in enumerate(['RNN', 'LSTM', 'GRU']):
    f1_scores = all_metrics[model_type]['per_class_f1']
    bars = ax.bar(x + i * width, f1_scores, width, 
                  label=model_type, color=COLORS[model_type], 
                  edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Per-Class F1 Score: RNN vs LSTM vs GRU', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels([f'Class {i+1}\n{CLASS_NAMES[i+1]}' for i in range(5)], fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3, label='_nolegend_')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves (one-vs-rest, per class)
from sklearn.preprocessing import label_binarize

# Binarize test labels
y_test_bin = label_binarize(all_test_labels, classes=list(range(5)))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    probs = all_probs_dict[model_type]
    
    for cls in range(5):
        fpr, tpr, _ = roc_curve(y_test_bin[:, cls], probs[:, cls])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=CLASS_COLORS[cls + 1], linewidth=1.5,
                label=f'C{cls+1}: {CLASS_NAMES[cls+1]} (AUC={roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title(f'{model_type} — One-vs-Rest ROC', fontsize=13,
                 color=COLORS[model_type], fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('ROC Curves per Class (One-vs-Rest)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Overall metrics table: Accuracy, Macro-F1, Weighted-F1, Params, Time
summary_rows = []
for model_type in ['RNN', 'LSTM', 'GRU']:
    cfg = best_configs[model_type]
    m = all_metrics[model_type]
    summary_rows.append({
        'Model': model_type,
        'Accuracy': f'{m["accuracy"]:.4f}',
        'Macro-F1': f'{m["macro_f1"]:.4f}',
        'Weighted-F1': f'{m["weighted_f1"]:.4f}',
        'Parameters': f'{final_models[model_type].count_parameters():,}',
        'Train Time (s)': f'{final_times[model_type]:.1f}',
        'Hidden': int(cfg['hidden_size']),
        'Layers': int(cfg['num_layers']),
        'Bidirectional': cfg['bidirectional']
    })

summary_df = pd.DataFrame(summary_rows)
print('\n' + '=' * 100)
print('FINAL COMPARISON TABLE (Test Set)')
print('=' * 100)
print(summary_df.to_string(index=False))

In [ ]:
# 1D-CNN Baseline
# Conv1d -> ReLU -> MaxPool -> Conv1d -> ReLU -> MaxPool -> Flatten -> Linear -> 5

class CNN1DClassifier(nn.Module):
    """1D Convolutional Neural Network baseline for ECG classification."""
    
    def __init__(self, input_length=140, num_classes=5):
        super().__init__()
        
        # Conv block 1: (batch, 1, 140) -> (batch, 32, 70)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        # Conv block 2: (batch, 32, 70) -> (batch, 64, 35)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        
        # Calculate flattened size
        self._flat_size = 64 * (input_length // 4)  # After two MaxPool(2): 140 -> 70 -> 35
        
        # Classification head
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self._flat_size, num_classes)
    
    def forward(self, x):
        # x: (batch, 140, 1) -> transpose to (batch, 1, 140) for Conv1d
        x = x.transpose(1, 2)  # (batch, 1, 140)
        
        x = self.pool1(self.relu1(self.conv1(x)))  # (batch, 32, 70)
        x = self.pool2(self.relu2(self.conv2(x)))  # (batch, 64, 35)
        
        x = self.flatten(x)    # (batch, 64*35=2240)
        x = self.dropout(x)
        x = self.fc(x)         # (batch, 5)
        return x
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Train CNN baseline
print('Training 1D-CNN Baseline...')
cnn_model = CNN1DClassifier(input_length=140, num_classes=5)
print(f'CNN Parameters: {cnn_model.count_parameters():,}')

cnn_history, cnn_time = train_model(
    cnn_model, train_loader, val_loader,
    epochs=FINAL_EPOCHS, lr=1e-3,
    class_weights=class_weights_tensor,
    patience=15, verbose=True
)

# Evaluate CNN
cnn_metrics, cnn_cm, cnn_preds, cnn_probs, _ = evaluate_classifier(
    cnn_model, test_loader, return_probs=True
)

print(f'\n1D-CNN Results:')
print(f'  Accuracy:    {cnn_metrics["accuracy"]:.4f}')
print(f'  Macro-F1:    {cnn_metrics["macro_f1"]:.4f}')
print(f'  Weighted-F1: {cnn_metrics["weighted_f1"]:.4f}')
print(f'  Parameters:  {cnn_model.count_parameters():,}')
print(f'  Train Time:  {cnn_time:.1f}s')

In [ ]:
# Bidirectional vs unidirectional comparison table
print('Bidirectional vs Unidirectional Comparison')
print('=' * 80)

bidir_results = []
for model_type in ['RNN', 'LSTM', 'GRU']:
    for bi in [False, True]:
        model = SignalClassifier(
            model_type=model_type, input_size=1, hidden_size=64,
            num_classes=5, num_layers=2, dropout=0.1, bidirectional=bi
        )
        
        history, t = train_model(
            model, train_loader, val_loader,
            epochs=40, lr=1e-3,
            class_weights=class_weights_tensor,
            patience=10, verbose=False
        )
        
        metrics, _, _ = evaluate_classifier(model, test_loader)
        
        bi_label = 'Bidirectional' if bi else 'Unidirectional'
        bidir_results.append({
            'Model': model_type,
            'Direction': bi_label,
            'Accuracy': f'{metrics["accuracy"]:.4f}',
            'Macro-F1': f'{metrics["macro_f1"]:.4f}',
            'Params': f'{model.count_parameters():,}',
            'Time (s)': f'{t:.1f}'
        })
        print(f'  {model_type:5s} {bi_label:15s}: Acc={metrics["accuracy"]:.4f}, '
              f'F1={metrics["macro_f1"]:.4f}, Params={model.count_parameters():,}')

bidir_df = pd.DataFrame(bidir_results)
print('\n')
print(bidir_df.to_string(index=False))

In [ ]:
# Combined comparison: RNN vs LSTM vs GRU vs 1D-CNN
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

all_model_names = ['RNN', 'LSTM', 'GRU', '1D-CNN']
all_model_colors = [COLORS['RNN'], COLORS['LSTM'], COLORS['GRU'], '#9b59b6']

# Collect all metrics including CNN
accs = [float(all_metrics[m]['accuracy']) for m in ['RNN', 'LSTM', 'GRU']] + [cnn_metrics['accuracy']]
f1s = [float(all_metrics[m]['macro_f1']) for m in ['RNN', 'LSTM', 'GRU']] + [cnn_metrics['macro_f1']]
params = [final_models[m].count_parameters() for m in ['RNN', 'LSTM', 'GRU']] + [cnn_model.count_parameters()]
times = [final_times[m] for m in ['RNN', 'LSTM', 'GRU']] + [cnn_time]

# 1. Accuracy
bars = axes[0].bar(all_model_names, accs, color=all_model_colors, edgecolor='white')
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Accuracy', fontsize=13)
axes[0].set_ylim(min(accs) * 0.95, max(accs) * 1.05)

# 2. Macro-F1
bars = axes[1].bar(all_model_names, f1s, color=all_model_colors, edgecolor='white')
for bar, val in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Macro-F1', fontsize=13)
axes[1].set_ylim(min(f1s) * 0.95, max(f1s) * 1.05)

# 3. Parameters
bars = axes[2].bar(all_model_names, params, color=all_model_colors, edgecolor='white')
for bar, val in zip(bars, params):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=8)
axes[2].set_title('Parameters', fontsize=13)

# 4. Training Time
bars = axes[3].bar(all_model_names, times, color=all_model_colors, edgecolor='white')
for bar, val in zip(bars, times):
    axes[3].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}s', ha='center', fontweight='bold', fontsize=9)
axes[3].set_title('Training Time', fontsize=13)

plt.suptitle('Final Model Comparison: RNN vs LSTM vs GRU vs 1D-CNN', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Analysis

In [ ]:
# Which classes are confused (from confusion matrix)
# Analyze the LSTM confusion matrix in detail
print('Confusion Matrix Analysis (LSTM)')
print('=' * 70)

cm_lstm = all_cms['LSTM']
cm_norm = cm_lstm.astype('float') / cm_lstm.sum(axis=1, keepdims=True)

print('\nNormalized confusion matrix (rows = true, columns = predicted):')
cm_df = pd.DataFrame(
    cm_norm,
    index=[f'True C{i+1}: {CLASS_NAMES[i+1]}' for i in range(5)],
    columns=[f'Pred C{i+1}' for i in range(5)]
)
print(cm_df.round(3).to_string())

print('\nKey Confusions:')
for i in range(5):
    for j in range(5):
        if i != j and cm_norm[i, j] > 0.05:
            print(f'  Class {i+1} ({CLASS_NAMES[i+1]}) misclassified as '
                  f'Class {j+1} ({CLASS_NAMES[j+1]}): {cm_norm[i, j]*100:.1f}%')

# Visualize confusion patterns
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Reds', ax=ax,
            xticklabels=[f'C{i+1}: {CLASS_NAMES[i+1]}' for i in range(5)],
            yticklabels=[f'C{i+1}: {CLASS_NAMES[i+1]}' for i in range(5)])
ax.set_title('LSTM Normalized Confusion Matrix', fontsize=14)
ax.set_xlabel('Predicted Class', fontsize=12)
ax.set_ylabel('True Class', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Misclassified heartbeat visualization (overlay with class average)
# Find misclassified samples from LSTM
lstm_preds = all_preds_dict['LSTM']
misclassified_mask = lstm_preds != all_test_labels
misclassified_indices = np.where(misclassified_mask)[0]

print(f'Total test samples: {len(all_test_labels)}')
print(f'Misclassified: {len(misclassified_indices)} ({len(misclassified_indices)/len(all_test_labels)*100:.1f}%)')

# Show up to 8 misclassified examples
n_show = min(8, len(misclassified_indices))
np.random.seed(SEED)
show_indices = np.random.choice(misclassified_indices, size=n_show, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for idx_in_plot, test_idx in enumerate(show_indices):
    ax = axes[idx_in_plot // 4, idx_in_plot % 4]
    
    true_cls = all_test_labels[test_idx]  # 0-indexed
    pred_cls = lstm_preds[test_idx]       # 0-indexed
    
    # Get the actual signal from test set
    signal = X_test_scaled[test_idx]
    
    # Compute class averages from scaled training data
    true_avg = X_train_scaled[y_train == true_cls].mean(axis=0)
    pred_avg = X_train_scaled[y_train == pred_cls].mean(axis=0)
    
    ax.plot(signal, color='black', linewidth=1.5, label='Sample', alpha=0.9)
    ax.plot(true_avg, color=CLASS_COLORS[true_cls + 1], linewidth=1.0,
            linestyle='--', alpha=0.7, label=f'Avg C{true_cls+1}')
    ax.plot(pred_avg, color=CLASS_COLORS[pred_cls + 1], linewidth=1.0,
            linestyle=':', alpha=0.7, label=f'Avg C{pred_cls+1}')
    
    ax.set_title(f'True: C{true_cls+1} ({CLASS_NAMES[true_cls+1]})\n'
                 f'Pred: C{pred_cls+1} ({CLASS_NAMES[pred_cls+1]})',
                 fontsize=9, color='red')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xlabel('Time Step', fontsize=8)

plt.suptitle('Misclassified Heartbeats (LSTM) — Sample vs Class Averages', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE of LSTM hidden states colored by class
print('Extracting LSTM hidden states for t-SNE visualization...')

# Hook to capture hidden states
lstm_model = final_models['LSTM'].to(DEVICE)
lstm_model.eval()

hidden_states_list = []
labels_list = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        # Get RNN output (all timesteps)
        rnn_out, _ = lstm_model.rnn(X_batch)
        # Take last hidden state
        last_hidden = rnn_out[:, -1, :].cpu().numpy()
        hidden_states_list.append(last_hidden)
        labels_list.extend(y_batch.numpy())

hidden_states = np.concatenate(hidden_states_list, axis=0)
labels_arr = np.array(labels_list)

print(f'Hidden states shape: {hidden_states.shape}')

# t-SNE
tsne_hidden = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000, verbose=0)
hidden_tsne = tsne_hidden.fit_transform(hidden_states)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# t-SNE of raw signals (from earlier)
# Re-compute for test set only
tsne_raw = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000, verbose=0)
raw_tsne = tsne_raw.fit_transform(X_test_scaled)

for cls in range(5):
    mask = labels_arr == cls
    ax1.scatter(raw_tsne[mask, 0], raw_tsne[mask, 1],
               c=CLASS_COLORS[cls + 1], label=f'C{cls+1}: {CLASS_NAMES[cls+1]}',
               s=15, alpha=0.6, edgecolors='none')
    ax2.scatter(hidden_tsne[mask, 0], hidden_tsne[mask, 1],
               c=CLASS_COLORS[cls + 1], label=f'C{cls+1}: {CLASS_NAMES[cls+1]}',
               s=15, alpha=0.6, edgecolors='none')

ax1.set_title('t-SNE of Raw Signals (140 dims)', fontsize=13)
ax1.set_xlabel('t-SNE 1', fontsize=11)
ax1.set_ylabel('t-SNE 2', fontsize=11)
ax1.legend(fontsize=9, markerscale=3)

ax2.set_title('t-SNE of LSTM Hidden States', fontsize=13)
ax2.set_xlabel('t-SNE 1', fontsize=11)
ax2.set_ylabel('t-SNE 2', fontsize=11)
ax2.legend(fontsize=9, markerscale=3)

plt.suptitle('Feature Space: Raw Signals vs LSTM Learned Representations', fontsize=14)
plt.tight_layout()
plt.show()

print('Observations:')
print('- LSTM hidden states show better class separation than raw signals')
print('- The recurrent network learns a more discriminative representation')
print('- Some overlap persists, especially among rare classes')

In [ ]:
# Gradient-based saliency: which timesteps matter most
print('Computing gradient-based saliency maps...')

def compute_saliency(model, X, y, device=DEVICE):
    """Compute gradient-based saliency for input signals.
    
    Returns:
        saliency: numpy array of shape (N, seq_len) - absolute gradient magnitude
    """
    model.eval()
    X_input = X.clone().detach().requires_grad_(True).to(device)
    y_target = y.clone().detach().to(device)
    
    logits = model(X_input)
    loss = F.cross_entropy(logits, y_target)
    loss.backward()
    
    # Absolute gradient w.r.t. input
    saliency = X_input.grad.abs().squeeze(-1).cpu().numpy()  # (N, 140)
    return saliency


# Compute saliency for each model
saliency_maps = {}
for model_type in ['RNN', 'LSTM', 'GRU']:
    model = final_models[model_type]
    saliency = compute_saliency(model, X_test_tensor, y_test_tensor)
    saliency_maps[model_type] = saliency
    print(f'  {model_type} saliency shape: {saliency.shape}')

# Average saliency per class for LSTM
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Average saliency across all classes per model
ax = axes[0]
for model_type in ['RNN', 'LSTM', 'GRU']:
    avg_saliency = saliency_maps[model_type].mean(axis=0)
    # Smooth with moving average
    window = 5
    smoothed = np.convolve(avg_saliency, np.ones(window)/window, mode='same')
    ax.plot(smoothed, color=COLORS[model_type], linewidth=1.5, label=model_type, alpha=0.8)

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Average |Gradient|', fontsize=12)
ax.set_title('Gradient Saliency: Which Timesteps Matter Most?', fontsize=14)
ax.legend(fontsize=11)
ax.axhline(0, color='gray', linewidth=0.5)

# Plot 2: Per-class saliency for LSTM
ax = axes[1]
for cls in range(5):
    mask = y_test == cls
    cls_saliency = saliency_maps['LSTM'][mask].mean(axis=0)
    smoothed = np.convolve(cls_saliency, np.ones(window)/window, mode='same')
    ax.plot(smoothed, color=CLASS_COLORS[cls + 1], linewidth=1.5,
            label=f'C{cls+1}: {CLASS_NAMES[cls+1]}')

ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Average |Gradient|', fontsize=12)
ax.set_title('LSTM Per-Class Saliency: Diagnostic Signal Regions', fontsize=14)
ax.legend(fontsize=9, loc='upper right')
ax.axhline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

print('Observations:')
print('- Saliency peaks indicate timesteps most important for classification')
print('- The QRS complex region typically shows highest saliency')
print('- Different classes have different important regions, matching clinical knowledge')
print('- LSTM and GRU focus on similar regions, while RNN may differ due to gradient limitations')

---
## 9. Conclusion

### Summary of Results

| Aspect | RNN | LSTM | GRU | 1D-CNN |
|--------|-----|------|-----|--------|
| **Architecture** | Simple recurrence | 3 gates (forget, input, output) | 2 gates (reset, update) | Convolutional filters |
| **Parameters** | Fewest (recurrent) | Most (recurrent) | Middle (recurrent) | Varies |
| **Strengths** | Fast, simple | Long-range dependencies | Good default, efficient | Local pattern detection |
| **Weaknesses** | Vanishing gradients | Slower training | Slightly less expressive | No temporal memory |

### Key Findings for ECG5000

1. **Class Imbalance Matters**: Class 1 (Normal) dominates with ~59% of samples. Without balanced sampling and weighted loss, models would simply predict "Normal" for everything.

2. **LSTM/GRU vs RNN**: On 140-timestep ECG signals, LSTM and GRU generally outperform vanilla RNN, as the gating mechanisms help capture the full morphology of heartbeat waveforms.

3. **1D-CNN as Strong Baseline**: Convolutional models are competitive because ECG patterns are often characterized by local morphological features (QRS complex shape, T-wave anomalies).

4. **Bidirectional Processing**: Slightly improves performance on ECG signals since the model can consider both past and future context within a single heartbeat window.

5. **Minority Class Challenges**: Rare arrhythmia classes (3, 4, 5) remain harder to classify accurately, which is a common challenge in medical signal classification.

6. **Saliency Analysis**: Gradient-based saliency reveals that models focus on the QRS complex and T-wave regions, which aligns with clinical diagnostic criteria.

### Extensions

- **Attention Mechanisms**: Add temporal attention to explicitly weight important timesteps
- **Transformer-based Models**: Self-attention for time series classification
- **Data Augmentation**: Time warping, jittering, magnitude scaling for minority classes
- **Multi-scale Features**: Combine raw signal with wavelet coefficients or frequency features
- **Ensemble Methods**: Combine RNN/LSTM/GRU/CNN predictions via voting or stacking
- **Transfer Learning**: Pre-train on larger ECG datasets (MIT-BIH, PTB-XL) and fine-tune
- **Real-time Deployment**: Optimize for edge devices (smartwatches, embedded systems)

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

model_names = ['RNN', 'LSTM', 'GRU']
model_colors = [COLORS[m] for m in model_names]

# 1. Accuracy comparison
accs = [all_metrics[m]['accuracy'] for m in model_names]
bars = axes[0].bar(model_names, accs, color=model_colors, edgecolor='white')
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title('Test Accuracy', fontsize=14)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim(min(accs) * 0.95, 1.0)

# 2. Macro-F1
f1s = [all_metrics[m]['macro_f1'] for m in model_names]
bars = axes[1].bar(model_names, f1s, color=model_colors, edgecolor='white')
for bar, val in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Test Macro-F1', fontsize=14)
axes[1].set_ylabel('Macro-F1', fontsize=12)
axes[1].set_ylim(min(f1s) * 0.95, 1.0)

# 3. Training time
times = [final_times[m] for m in model_names]
bars = axes[2].bar(model_names, times, color=model_colors, edgecolor='white')
for bar, val in zip(bars, times):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}s', ha='center', fontweight='bold', fontsize=11)
axes[2].set_title('Training Time', fontsize=14)
axes[2].set_ylabel('Seconds', fontsize=12)

plt.suptitle('Final Summary: ECG5000 Heartbeat Classification', fontsize=15)
plt.tight_layout()
plt.show()

# Print winner
best_model = max(model_names, key=lambda m: all_metrics[m]['macro_f1'])
print(f'\nBest model by Macro-F1: {best_model} '
      f'(F1={all_metrics[best_model]["macro_f1"]:.4f}, '
      f'Acc={all_metrics[best_model]["accuracy"]:.4f})')
print('\nNotebook complete! Case Study 8 - ECG5000 Time Series Classification.')